Mục tiêu: Check Data Integrity của Dataset 

Ngày phân tích: 2026-08-06



Step 1: Reload Data

Note: Không có giá trị Null trong dataset này

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../data/cookie_cats.csv")

df = pd.read_csv(
    DATA_PATH,
    dtype={
        "userid": "int64",
        "version": "string",
        "sum_gamerounds": "int64",
        "retention_1": "bool",
        "retention_7": "bool",
    },
)

print(f"Shape: {df.shape}")
df.info()

Shape: (90189, 5)
<class 'pandas.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   userid          90189 non-null  int64 
 1   version         90189 non-null  string
 2   sum_gamerounds  90189 non-null  int64 
 3   retention_1     90189 non-null  bool  
 4   retention_7     90189 non-null  bool  
dtypes: bool(2), int64(2), string(1)
memory usage: 2.2 MB


Step 2: Duplicate check

Note: Dataset không tồn tại duplicate

In [2]:
n_total = len(df)
n_unique_users = df["userid"].nunique()
n_duplicated_users = n_total - n_unique_users

print(f"Total rows:      {n_total:,}")
print(f"Unique userids:  {n_unique_users:,}")
print(f"Duplicate count: {n_duplicated_users:,}")

Total rows:      90,189
Unique userids:  90,189
Duplicate count: 0


Step 2.1: Deduplication policy

Policy: Nếu userid trùng, giữ record có sum_gamerounds cao nhất.

Rationale: Giả định record cao hơn = data mới hơn (user chơi thêm rounds).

Note: Dataset không có Duplicates

In [8]:
n_before = len(df)
n_duplicates_before = n_before - df["userid"].nunique()

if n_duplicates_before > 0:
    # Sort giảm dần theo sum_gamerounds, giữ record đầu tiên (cao nhất) cho mỗi userid
    df = (
        df.sort_values("sum_gamerounds", ascending=False)
          .drop_duplicates(subset="userid", keep="first")
          .sort_values("userid")   # sort lại theo userid cho dễ đọc
          .reset_index(drop=True)
    )
    n_after = len(df)
    n_removed = n_before - n_after
    print(f"   Dedup applied:")
    print(f"   Rows before: {n_before:,}")
    print(f"   Rows after:  {n_after:,}")
    print(f"   Removed:     {n_removed:,}")
else:
    print(f" No duplicates to remove ({n_before:,} rows unchanged)")

# Sanity check hậu-dedup — chạy dù có dedup hay không
assert df["userid"].is_unique, "Dedup failed: still have duplicate userids"
print(f"\n Post-dedup: {len(df):,} rows; \n Unique users: {df['userid'].nunique():,} ")

 No duplicates to remove (90,189 rows unchanged)

 Post-dedup: 90,189 rows; 
 Unique users: 90,189 


Step 3: Null check (Trong trường hợp tồn tại Null record)

Note: Null record không tồn tại trong record này

In [8]:
null_summary = pd.DataFrame({
    "null_count": df.isnull().sum(),
    "null_pct": (df.isnull().sum() / len(df) * 100).round(4),
    "dtype": df.dtypes.astype(str),
})
print(null_summary)

                null_count  null_pct   dtype
userid                   0       0.0   int64
version                  0       0.0  string
sum_gamerounds           0       0.0   int64
retention_1              0       0.0    bool
retention_7              0       0.0    bool


Step 4: Value check cho cột "Version"

Note: "Version" hoàn toàn sạch

In [12]:
version_counts = df["version"].value_counts(dropna=False)
print("Version Distribution:")
print(version_counts)
print()
print(f"Unique versions: {df['version'].unique()}")
print(f"Distinct values: {df['version'].unique().tolist()}")

# Check whitespace/case bugs (rất hay gặp trong data thật)
version_stripped = df["version"].str.strip().str.lower()
if not (version_stripped == df["version"]).all():
    print("Whitespace hoặc case inconsistency detected")
else:
    print("Version values are clean")

Version Distribution:
version
gate_40    45489
gate_30    44700
Name: count, dtype: Int64

Unique versions: <StringArray>
['gate_30', 'gate_40']
Length: 2, dtype: string
Distinct values: ['gate_30', 'gate_40']
Version values are clean


Step 5: Sanity check cho sum_gamerounds

Note: giá trị Max vượt xa mean và median => cần check lại outliers ở nhóm 4; không có negative values

In [13]:
sr = df["sum_gamerounds"]
print(f"Min:    {sr.min()}")
print(f"Max:    {sr.max():,}")
print(f"Median: {sr.median()}")
print(f"Mean:   {sr.mean():.2f}")

# Sum_gamerounds không được âm
if (sr < 0).any():
    print(f"⚠️ {(sr < 0).sum()} rows có sum_gamerounds âm — impossible")
else:
    print("✓ No negative values")

# Bao nhiêu user không chơi round nào (install rồi bỏ ngay)
n_zero = (sr == 0).sum()
print(f"\nUsers với 0 rounds: {n_zero:,} ({n_zero/len(df)*100:.2f}%)")

Min:    0
Max:    49,854
Median: 16.0
Mean:   51.87
✓ No negative values

Users với 0 rounds: 3,994 (4.43%)


Step 6: Logic consistency check

In [ ]:
# Kiểm tra: user có sum_gamerounds = 0 mà retention_1 = True hoặc retention_7 = True
# → Impossible: không chơi round nào thì tại sao họ lại "quay lại"?
impossible_1 = df[(df["sum_gamerounds"] == 0) & (df["retention_1"] == True)]
impossible_7 = df[(df["sum_gamerounds"] == 0) & (df["retention_7"] == True)]

print(f"Users với 0 rounds nhưng retention_1=True: {len(impossible_1)}")
print(f"Users với 0 rounds nhưng retention_7=True: {len(impossible_7)}")

# Kiểm tra: retention_7 = True mà retention_1 = False
# → Về mặt logic vẫn possible (user không chơi D1 nhưng chơi D7)
# → Nhưng đáng để đếm và document
r7_not_r1 = df[(df["retention_7"] == True) & (df["retention_1"] == False)]
print(f"\nUsers retention_7=True nhưng retention_1=False: {len(r7_not_r1):,} ({len(r7_not_r1)/len(df)*100:.2f}%)")
print("→ Số này hợp lệ về logic (user skip D1 rồi quay lại D7), nhưng đáng để quan tâm.")

Users với 0 rounds nhưng retention_1=True: 87
Users với 0 rounds nhưng retention_7=True: 29

Users retention_7=True nhưng retention_1=False: 3,599 (3.99%)
→ Số này hợp lệ về logic (user skip D1 rồi quay lại D7), nhưng đáng note.
